In [1]:
import os
import sys
import joblib
import glob
import pandas as pd
from pathlib import Path
 
sys.path.insert(0, '../../..')  # Load Parent directory to import utils
from utils.constants import Configs

RESULTS_DIR = Configs._wrist_only_loso_results_cache_dir

In [2]:
def load_all_fold_results(results_dir):
    rows = []
    for file in glob.glob(os.path.join(results_dir, "*.pkl")):
        obj = joblib.load(file)
        if isinstance(obj, dict) and "Fold_History" in obj:
            df = pd.DataFrame(obj["Fold_History"])
            stem = Path(file).stem.replace("wrist_only_loso_results_", "")
            model, representation, phase = stem.rsplit("_", 2)
            df["Model"] = model
            df["Representation"] = representation
            df["Phase"] = phase
            rows.append(df)
        elif isinstance(obj, pd.DataFrame):
            rows.append(obj)
        else:
            raise TypeError(f"{file} contains {type(obj).__name__}, not a supported result format")
    return pd.concat(rows, ignore_index=True)
 

In [3]:
consolidated_data = load_all_fold_results(RESULTS_DIR)
print(f"Successfully loaded {len(consolidated_data)} fold records.")
consolidated_data.head()

Successfully loaded 418 fold records.


,subject_out,params,metrics,Model,Representation,Phase
0,1,{'z_threshold': 2.0},"{'AUPRC': nan, 'AUROC': nan, 'TP': 0, 'FP': 0,...",ZScore,ShapeBased,Move
1,2,{'z_threshold': 2.0},"{'AUPRC': nan, 'AUROC': nan, 'TP': 1, 'FP': 0,...",ZScore,ShapeBased,Move
2,3,{'z_threshold': 2.0},"{'AUPRC': nan, 'AUROC': nan, 'TP': 0, 'FP': 0,...",ZScore,ShapeBased,Move
3,4,{'z_threshold': 2.0},"{'AUPRC': nan, 'AUROC': nan, 'TP': 1, 'FP': 0,...",ZScore,ShapeBased,Move
4,5,{'z_threshold': 2.5},"{'AUPRC': nan, 'AUROC': nan, 'TP': 0, 'FP': 0,...",ZScore,ShapeBased,Move


### Filter Especific LOSO

In [4]:
DESIRED_REPRESENTATION = "FeatureBased"
DESIRED_MODEL = "DBSCAN"
DESIRED_PHASE = "Pick"
desired_loso = consolidated_data[
    (consolidated_data["Representation"] == DESIRED_REPRESENTATION) &
    (consolidated_data["Model"] == DESIRED_MODEL) &
    (consolidated_data["Phase"] == DESIRED_PHASE)
]
desired_metrics_df = pd.DataFrame(list(desired_loso["metrics"]))
desired_metrics_df

,AUPRC,AUROC,TP,FP,FN,TN,Precision,Recall,F1,F2,MCC,Pct_Rejection,N_Known_Positives,N_Predicted_Positives,Scores
0,NaN,NaN,0,1,0,19,NaN,NaN,NaN,NaN,NaN,5.0,0,1,None
1,NaN,NaN,0,1,0,19,NaN,NaN,NaN,NaN,NaN,5.0,0,1,None
2,NaN,NaN,0,0,0,20,NaN,NaN,NaN,NaN,NaN,0.0,0,0,None
3,NaN,NaN,0,0,1,19,0.000000,0.0,0.000000,0.000000,0.000000,0.0,1,0,None
4,NaN,NaN,1,0,1,18,1.000000,0.5,0.666667,0.555556,0.688247,5.0,2,1,None
5,NaN,NaN,0,0,0,20,NaN,NaN,NaN,NaN,NaN,0.0,0,0,None
6,NaN,NaN,0,0,0,20,NaN,NaN,NaN,NaN,NaN,0.0,0,0,None
7,NaN,NaN,1,0,0,19,1.000000,1.0,1.000000,1.000000,1.000000,5.0,1,1,None
8,NaN,NaN,1,0,0,19,1.000000,1.0,1.000000,1.000000,1.000000,5.0,1,1,None
9,NaN,NaN,0,1,0,19,NaN,NaN,NaN,NaN,NaN,5.0,0,1,None
